# 09 — Ablation Training Runs — **FIXED**

Trains **BanglaBERT only, seed=42** under 5 conditions.
Results feed into Table 2 in NB07.

**Fixes vs original:**
- Loads `label_encoders.json` from NB05 (no more int/str key mismatch)
- `sexual,religious → sexual` (matches NB05 priority)
- Model architecture matches NB05 exactly (TaskHead 2-layer, CLS+mean pooling)
- Fixed `n_layers` calculation and `GradScaler` API
- All conditions now actually train (binary labels were silently all -1 before)

**Requires GPU. ~4–5 hrs total. Run after NB05.**


In [1]:
import os, json, copy, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, accuracy_score, matthews_corrcoef
from collections import defaultdict
import math, time, random

warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
os.makedirs('../outputs/ablations', exist_ok=True)


Device: cuda
GPU: NVIDIA GeForce RTX 4060 Ti


## Load data + label consolidation (identical to NB05)

In [2]:
SPLITS_DIR = '../data/splits'
MODELS_DIR = '../outputs/models_v2_fix'

train_df = pd.read_csv(f'{SPLITS_DIR}/random_train.csv')
val_df   = pd.read_csv(f'{SPLITS_DIR}/random_val.csv')
test_df  = pd.read_csv(f'{SPLITS_DIR}/random_test.csv')

# ── Identical consolidation to NB05 ──────────────────────────────────
TYPE_MAP = {
    'none':'none','not bully':'none',
    'threat':'threat','threat,spam':'threat',
    'callToViolence':'threat','callToViolence_slander':'threat',
    'callToViolence_gender':'threat','callToViolence_religion':'threat',
    'callToViolence_religion_slander':'threat',
    'callToViolence_gender_religion_slander':'threat',
    'callToViolence_gender_slander':'threat',
    'religious,threat':'threat','sexual,threat':'threat','sexual,religious,threat':'threat',
    'sexual':'sexual',
    'sexual,religious':'sexual',   # FIX: sexual > religious
    'sexual,spam':'sexual',
    'religious':'religious','Religious':'religious','religion':'religious',
    'religious,spam':'religious','religion_slander':'religious',
    'gender_religion':'religious','gender_religion_slander':'religious',
    'gender':'gender','Gender':'gender','gender_slander':'gender',
    'Political':'political',
    'Personal Offense':'personal','Body Shaming':'personal',
    'Origin':'personal','slander':'personal','Misc':'personal',
    'Abusive/Violence':'abusive','troll':'abusive',
    'spam':'other',
}
PRIORITY = ['threat','sexual','religious','gender','political','abusive','personal','other','none']

def consolidate_type(val):
    if not isinstance(val, str) or not val.strip(): return 'none'
    val = val.strip()
    if val in TYPE_MAP: return TYPE_MAP[val]
    parts = [p.strip() for p in val.replace(';',',').split(',')]
    cands = [TYPE_MAP[p] for p in parts if p in TYPE_MAP]
    if cands:
        for pc in PRIORITY:
            if pc in cands: return pc
    sl = val.lower().replace('_',' ')
    for kw, cls in [('threat','threat'),('calltoviolence','threat'),('sexual','sexual'),
                    ('religious','religious'),('religion','religious'),('gender','gender'),
                    ('political','political'),('abusive','abusive'),('violence','abusive'),
                    ('personal','personal'),('slander','personal'),('origin','personal'),
                    ('body','personal'),('misc','personal'),('spam','other')]:
        if kw in sl.replace(' ',''): return cls
    return 'other'

for df_ in [train_df, val_df, test_df]:
    df_['label_type'] = df_['label_type'].apply(consolidate_type)

TEXT_COL = 'text_clean' if 'text_clean' in train_df.columns else 'text'
print(f'Train: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}')
print(f'Text col: {TEXT_COL}')
print(f'abuse_type classes in train: {sorted(train_df["label_type"].unique())}')


Train: 108,460  Val: 13,557  Test: 13,558
Text col: text_clean
abuse_type classes in train: ['abusive', 'gender', 'none', 'other', 'personal', 'political', 'religious', 'sexual', 'threat']


## Load label encoders from NB05

**Critical:** uses the same encoders NB05 trained with — no type mismatches possible.

In [3]:
LE_PATH = f'{MODELS_DIR}/label_encoders.json'
assert os.path.exists(LE_PATH), (
    f'label_encoders.json not found at {LE_PATH}. Run NB05 first.'
)
with open(LE_PATH) as _f:
    _raw = json.load(_f)

# NB05 saves binary keys as strings (JSON) — restore correct types
label_enc_full = {}
for task, enc in _raw.items():
    if task == 'binary':
        label_enc_full[task] = {int(k): v for k, v in enc.items()}
    else:
        label_enc_full[task] = enc

# binary-only encoder (subset)
label_enc_binary = {'binary': label_enc_full['binary']}

# Task configs: num_classes comes from encoder size
TASK_CONFIG_FULL = {
    'binary':     {'col': 'label_binary', 'num_classes': len(label_enc_full['binary']),     'loss_weight': 1.0},
    'abuse_type': {'col': 'label_type',   'num_classes': len(label_enc_full['abuse_type']), 'loss_weight': 0.9},
}
TASK_CONFIG_BINARY = {
    'binary': {'col': 'label_binary', 'num_classes': len(label_enc_full['binary']), 'loss_weight': 1.0},
}

print('Label encoders loaded from NB05:')
for t, enc in label_enc_full.items():
    print(f'  {t}: {len(enc)} classes  key_type={type(next(iter(enc))).__name__}')
print(f'\nabuse_type mapping: {label_enc_full["abuse_type"]}')


Label encoders loaded from NB05:
  binary: 2 classes  key_type=int
  abuse_type: 9 classes  key_type=str

abuse_type mapping: {'abusive': 0, 'gender': 1, 'none': 2, 'other': 3, 'personal': 4, 'political': 5, 'religious': 6, 'sexual': 7, 'threat': 8}


## Model classes — identical to NB05

In [4]:
def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
    os.environ['PYTHONHASHSEED'] = str(s)


class AblationDataset(Dataset):
    def __init__(self, df, tokenizer, max_length, active_tasks, label_encoders, text_col):
        self.texts  = df.reset_index(drop=True)[text_col].fillna('').astype(str).tolist()
        self.tok    = tokenizer
        self.maxlen = max_length
        self.labels = {}
        for tname, tcfg in active_tasks.items():
            enc = label_encoders[tname]
            col = tcfg['col']
            # Keys in enc are int for binary, str for abuse_type — use the right lookup
            if tname == 'binary':
                self.labels[tname] = [
                    int(enc.get(int(v) if not pd.isna(v) else -1, -1))
                    for v in df[col].fillna(-1)
                ]
            else:
                self.labels[tname] = [
                    int(enc.get(str(v), -1))
                    for v in df[col].fillna('unknown')
                ]

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        enc  = self.tok(self.texts[idx], max_length=self.maxlen,
                        truncation=True, padding=False)
        item = {k: torch.tensor(v, dtype=torch.long) for k, v in enc.items()}
        for t in self.labels:
            item[f'label_{t}'] = torch.tensor(self.labels[t][idx], dtype=torch.long)
        return item


class AblationCollator:
    def __init__(self, tokenizer): self.tokenizer = tokenizer
    def __call__(self, features):
        lkeys  = [k for k in features[0] if k.startswith('label_')]
        labels = {k: torch.stack([f[k] for f in features]) for k in lkeys}
        tfeats = [{k: v for k, v in f.items() if not k.startswith('label_')} for f in features]
        batch  = self.tokenizer.pad(tfeats, padding=True, return_tensors='pt')
        batch.update(labels)
        return batch


class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None, label_smoothing=0.0):
        super().__init__()
        self.gamma, self.weight, self.ls = gamma, weight, label_smoothing
    def forward(self, logits, targets):
        ce   = F.cross_entropy(logits, targets, weight=self.weight,
                               reduction='none', label_smoothing=self.ls)
        loss = ((1 - torch.exp(-ce)) ** self.gamma) * ce
        return loss.mean()


class TaskHead(nn.Module):
    # Matches NB05 exactly: Dropout -> Linear -> GELU -> LayerNorm -> Dropout -> Linear
    def __init__(self, hidden, n_cls, dropout=0.25, inner=384):
        super().__init__()
        inner = min(inner, hidden)
        self.net = nn.Sequential(
            nn.Dropout(dropout), nn.Linear(hidden, inner),
            nn.GELU(), nn.LayerNorm(inner),
            nn.Dropout(dropout), nn.Linear(inner, n_cls),
        )
    def forward(self, x): return self.net(x)


class MultiTaskTransformer(nn.Module):
    def __init__(self, model_name, active_tasks, dropout=0.25, head_hidden_dim=384):
        super().__init__()
        self.encoder  = AutoModel.from_pretrained(model_name)
        hidden        = self.encoder.config.hidden_size
        mtype         = self.encoder.config.model_type.lower()
        self._use_tti = mtype not in ('roberta', 'xlm-roberta')
        self.heads    = nn.ModuleDict({
            t: TaskHead(hidden, cfg['num_classes'], dropout, head_hidden_dim)
            for t, cfg in active_tasks.items()
        })

    def _pool(self, out, mask):
        hs   = out.last_hidden_state
        cls  = hs[:, 0]
        mean = (hs * mask.unsqueeze(-1).float()).sum(1) / mask.sum(1, keepdim=True).float().clamp(1)
        return 0.5 * cls + 0.5 * mean   # CLS + mean blend, matches NB05

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        kw = {'input_ids': input_ids, 'attention_mask': attention_mask}
        if self._use_tti and token_type_ids is not None:
            kw['token_type_ids'] = token_type_ids
        out = self.encoder(**kw)
        p   = self._pool(out, attention_mask)
        return {t: h(p) for t, h in self.heads.items()}


def get_layerwise_params(model, enc_lr, head_lr, decay, wd):
    no_decay = ['bias', 'LayerNorm.weight', 'LayerNorm.bias']
    n_layers = 0
    for name, _ in model.encoder.named_parameters():
        for p in name.split('.'):
            if p.isdigit(): n_layers = max(n_layers, int(p)+1)
    n_layers = max(n_layers, 12)

    groups = []
    for name, param in model.encoder.named_parameters():
        if not param.requires_grad: continue
        ln = 0
        for p in name.split('.'):
            if p.isdigit(): ln = int(p); break
        _lr  = enc_lr * (decay ** (n_layers - ln - 1))
        _wd  = 0.0 if any(nd in name for nd in no_decay) else wd
        groups.append({'params': [param], 'lr': _lr, 'weight_decay': _wd})
    for name, param in model.heads.named_parameters():
        if not param.requires_grad: continue
        _wd = 0.0 if any(nd in name for nd in no_decay) else wd
        groups.append({'params': [param], 'lr': head_lr, 'weight_decay': _wd})
    return groups


def build_class_weights(series, enc, beta=0.999, max_w=10.0):
    mapped = series.map(enc).dropna().astype(int)
    n_cls  = len(enc)
    counts = mapped.value_counts().sort_index()
    w      = np.ones(n_cls, dtype=np.float32)
    for i in range(n_cls):
        n = int(counts.get(i, 0))
        if n > 0:
            eff  = (1.0 - beta**n) / max(1.0 - beta, 1e-12)
            w[i] = 1.0 / max(eff, 1e-9)
    w = np.clip(w, w.min(), w.min() * max_w)
    w = w / w.mean()
    return torch.tensor(w, dtype=torch.float32, device=device)


@torch.no_grad()
def evaluate(model, loader, active_tasks):
    model.eval()
    results = {}
    all_p, all_l = defaultdict(list), defaultdict(list)
    for batch in loader:
        batch  = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
        logits = model(batch['input_ids'], batch['attention_mask'],
                       batch.get('token_type_ids'))
        for t in active_tasks:
            lbl  = batch[f'label_{t}'].cpu().numpy()
            pred = logits[t].argmax(-1).cpu().numpy()
            m    = lbl >= 0
            all_p[t].extend(pred[m]); all_l[t].extend(lbl[m])
    for t in active_tasks:
        y, p = np.array(all_l[t]), np.array(all_p[t])
        results[t] = {
            'macro_f1':    round(float(f1_score(y, p, average='macro',    zero_division=0)), 4),
            'weighted_f1': round(float(f1_score(y, p, average='weighted', zero_division=0)), 4),
            'accuracy':    round(float(accuracy_score(y, p)), 4),
            'mcc':         round(float(matthews_corrcoef(y, p)), 4),
        }
    return results

print('All classes defined ✅')


All classes defined ✅


## Ablation conditions

| Tag | Change |
|---|---|
| `full_multitask` | Full NB05 system — BanglaBERT seed=42 (reference) |
| `binary_only` | Remove abuse_type head |
| `no_focal` | Standard CrossEntropy instead of FocalLoss |
| `no_lrdecay` | Uniform LR across all layers |
| `no_preprocessing` | Raw `text` column instead of `text_clean` |


In [5]:
BASE_MODEL     = 'csebuetnlp/banglabert'
BASE_MODEL_KEY = 'banglabert'
SEED           = 42

BASE_CONFIG = {
    'max_length':      128,
    'batch_size':      16,        # micro-batch
    'grad_accum':      2,         # effective batch = 32 (matches NB05)
    'eval_batch_size': 32,
    'epochs':          8,
    'encoder_lr':      2e-5,
    'head_lr':         8e-5,
    'weight_decay':    0.01,
    'warmup_ratio':    0.10,
    'lr_decay_factor': 0.90,
    'focal_gamma_binary':     1.5,
    'focal_gamma_abuse_type': 2.5,
    'label_smoothing': 0.03,
    'class_weight_beta': 0.999,
    'dropout':         0.25,
    'head_hidden_dim': 384,
    'patience':        3,
    'use_fp16':        torch.cuda.is_available(),
}
print('Config ready.')
print(f'Effective batch size: {BASE_CONFIG["batch_size"]} × {BASE_CONFIG["grad_accum"]} = '
      f'{BASE_CONFIG["batch_size"]*BASE_CONFIG["grad_accum"]}')


Config ready.
Effective batch size: 16 × 2 = 32


## Core training function

In [6]:
def run_ablation(tag, active_tasks, label_encoders, config,
                 use_focal=True, use_lrdecay=True, text_col='text_clean'):
    set_seed(SEED)
    save_dir = f'../outputs/ablations/{tag}'
    os.makedirs(save_dir, exist_ok=True)

    print(f'\n{"="*62}')
    print(f'  ABLATION: {tag}  |  {BASE_MODEL_KEY}  |  seed={SEED}')
    print(f'{"="*62}')

    tok      = AutoTokenizer.from_pretrained(BASE_MODEL)
    collator = AblationCollator(tok)
    lkw      = dict(collate_fn=collator, num_workers=0,
                    pin_memory=torch.cuda.is_available())

    train_loader = DataLoader(
        AblationDataset(train_df, tok, config['max_length'],
                        active_tasks, label_encoders, text_col),
        batch_size=config['batch_size'], shuffle=True, drop_last=True, **lkw)
    val_loader = DataLoader(
        AblationDataset(val_df, tok, config['max_length'],
                        active_tasks, label_encoders, text_col),
        batch_size=config['eval_batch_size'], shuffle=False, **lkw)
    test_loader = DataLoader(
        AblationDataset(test_df, tok, config['max_length'],
                        active_tasks, label_encoders, text_col),
        batch_size=config['eval_batch_size'], shuffle=False, **lkw)

    model = MultiTaskTransformer(
        BASE_MODEL, active_tasks,
        dropout=config['dropout'],
        head_hidden_dim=config['head_hidden_dim'],
    ).to(device)

    # ── Loss functions ────────────────────────────────────────────────
    criteria = {}
    for tname, tcfg in active_tasks.items():
        # Build class weights (safe clamped version)
        w = build_class_weights(train_df[tcfg['col']], label_encoders[tname],
                                 beta=config['class_weight_beta'])
        gamma = config.get(f'focal_gamma_{tname}', 2.0)
        if use_focal:
            criteria[tname] = FocalLoss(gamma=gamma, weight=w,
                                         label_smoothing=config['label_smoothing'])
        else:
            # no_focal ablation: standard weighted cross-entropy
            criteria[tname] = nn.CrossEntropyLoss(weight=w)

    # ── Optimizer ─────────────────────────────────────────────────────
    if use_lrdecay:
        param_groups = get_layerwise_params(
            model, config['encoder_lr'], config['head_lr'],
            config['lr_decay_factor'], config['weight_decay'])
    else:
        # no_lrdecay: uniform LR, no layer-wise decay
        no_decay = ['bias', 'LayerNorm.weight', 'LayerNorm.bias']
        param_groups = [
            {'params': [p for n,p in model.named_parameters()
                        if not any(nd in n for nd in no_decay)],
             'lr': config['encoder_lr'], 'weight_decay': config['weight_decay']},
            {'params': [p for n,p in model.named_parameters()
                        if any(nd in n for nd in no_decay)],
             'lr': config['encoder_lr'], 'weight_decay': 0.0},
        ]

    optimizer = torch.optim.AdamW(param_groups)
    n_steps   = math.ceil(len(train_loader) / config['grad_accum']) * config['epochs']
    scheduler = get_linear_schedule_with_warmup(
        optimizer, int(n_steps * config['warmup_ratio']), n_steps)
    scaler    = torch.amp.GradScaler('cuda') if (config['use_fp16'] and device.type=='cuda') else None

    best_score, no_improve = -1.0, 0

    for epoch in range(config['epochs']):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        ep_loss, t0 = 0.0, time.time()

        for step, batch in enumerate(train_loader, 1):
            batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
            with torch.autocast(device_type=device.type,
                                enabled=config['use_fp16'] and device.type=='cuda'):
                logits = model(batch['input_ids'], batch['attention_mask'],
                               batch.get('token_type_ids'))
                loss = torch.tensor(0.0, device=device)
                for tname, tcfg in active_tasks.items():
                    lbl = batch[f'label_{tname}']
                    vm  = lbl >= 0
                    if vm.any():
                        tl = criteria[tname](logits[tname][vm], lbl[vm])
                        if not (torch.isnan(tl) or torch.isinf(tl)):
                            loss = loss + tcfg['loss_weight'] * tl
                loss = loss / config['grad_accum']

            (scaler.scale(loss) if scaler else loss).backward()

            if step % config['grad_accum'] == 0 or step == len(train_loader):
                if scaler:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(optimizer); scaler.update()
                else:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optimizer.step()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

            ep_loss += loss.item() * config['grad_accum']

        avg_loss = ep_loss / max(len(train_loader), 1)
        val_m    = evaluate(model, val_loader, active_tasks)
        bin_f1   = val_m.get('binary', {}).get('macro_f1', 0.0)
        abu_f1   = val_m.get('abuse_type', {}).get('macro_f1', 0.0)
        monitor  = (active_tasks.get('binary', {}).get('loss_weight', 1.0) * bin_f1 * 0.7 +
                    active_tasks.get('abuse_type', {}).get('loss_weight', 0.0) * abu_f1 * 0.3
                    if 'abuse_type' in active_tasks else bin_f1)

        flag = ''
        if monitor > best_score:
            best_score = monitor; no_improve = 0
            torch.save(model.state_dict(), f'{save_dir}/best_model.pt'); flag = ' ✅BEST'
        else:
            no_improve += 1

        abu_str = f' abu={abu_f1:.4f}' if 'abuse_type' in active_tasks else ''
        print(f'  Ep{epoch+1:2}/{config["epochs"]} loss={avg_loss:.4f} '
              f'bin={bin_f1:.4f}{abu_str} {(time.time()-t0)/60:.1f}min{flag}')

        if no_improve >= config['patience']:
            print(f'  Early stop at epoch {epoch+1}'); break

    # ── Final test eval ───────────────────────────────────────────────
    model.load_state_dict(torch.load(f'{save_dir}/best_model.pt',
                                     map_location=device, weights_only=True))
    test_m = evaluate(model, test_loader, active_tasks)

    result = {
        'tag':            tag,
        'best_val_score': round(best_score, 4),
        'binary_macro_f1':    test_m['binary']['macro_f1'],
        'binary_accuracy':    test_m['binary']['accuracy'],
        'binary_mcc':         test_m['binary']['mcc'],
        'binary_weighted_f1': test_m['binary']['weighted_f1'],
    }
    if 'abuse_type' in test_m:
        result['abuse_type_macro_f1']    = test_m['abuse_type']['macro_f1']
        result['abuse_type_accuracy']    = test_m['abuse_type']['accuracy']
        result['abuse_type_mcc']         = test_m['abuse_type']['mcc']
        result['abuse_type_weighted_f1'] = test_m['abuse_type']['weighted_f1']

    with open(f'{save_dir}/results.json', 'w') as _f:
        json.dump(result, _f, indent=2)

    print(f'  Test → bin={result["binary_macro_f1"]:.4f}', end='')
    if 'abuse_type_macro_f1' in result:
        print(f'  abu={result["abuse_type_macro_f1"]:.4f}', end='')
    print()

    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return result

print('run_ablation defined ✅')


run_ablation defined ✅


## Run all ablations

Each condition saves results immediately — safe to interrupt and resume.

In [7]:
ablation_results = []

# ── 1. Full multi-task (reference) ────────────────────────────────────
cfg = copy.deepcopy(TASK_CONFIG_FULL)
r = run_ablation('full_multitask', cfg, label_enc_full, BASE_CONFIG,
                  use_focal=True, use_lrdecay=True, text_col=TEXT_COL)
ablation_results.append(r)
pd.DataFrame(ablation_results).to_csv('../outputs/ablations/ablation_results.csv', index=False)

# ── 2. Binary-only (no abuse_type head) ───────────────────────────────
cfg = copy.deepcopy(TASK_CONFIG_BINARY)
r = run_ablation('binary_only', cfg, label_enc_binary, BASE_CONFIG,
                  use_focal=True, use_lrdecay=True, text_col=TEXT_COL)
ablation_results.append(r)
pd.DataFrame(ablation_results).to_csv('../outputs/ablations/ablation_results.csv', index=False)

# ── 3. No focal loss (standard cross-entropy + class weights) ─────────
cfg = copy.deepcopy(TASK_CONFIG_FULL)
r = run_ablation('no_focal', cfg, label_enc_full, BASE_CONFIG,
                  use_focal=False, use_lrdecay=True, text_col=TEXT_COL)
ablation_results.append(r)
pd.DataFrame(ablation_results).to_csv('../outputs/ablations/ablation_results.csv', index=False)

# ── 4. No layer-wise LR decay (uniform LR) ────────────────────────────
cfg = copy.deepcopy(TASK_CONFIG_FULL)
r = run_ablation('no_lrdecay', cfg, label_enc_full, BASE_CONFIG,
                  use_focal=True, use_lrdecay=False, text_col=TEXT_COL)
ablation_results.append(r)
pd.DataFrame(ablation_results).to_csv('../outputs/ablations/ablation_results.csv', index=False)

# ── 5. No preprocessing (raw text instead of text_clean) ──────────────
cfg = copy.deepcopy(TASK_CONFIG_FULL)
r = run_ablation('no_preprocessing', cfg, label_enc_full, BASE_CONFIG,
                  use_focal=True, use_lrdecay=True, text_col='text')
ablation_results.append(r)
pd.DataFrame(ablation_results).to_csv('../outputs/ablations/ablation_results.csv', index=False)

print(f'\n🎉 All {len(ablation_results)} ablations complete')



  ABLATION: full_multitask  |  banglabert  |  seed=42


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

ElectraModel LOAD REPORT from: csebuetnlp/banglabert
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
discriminator_predictions.dense.weight            | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED |  | 
electra.embeddings.position_ids                   | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED |  | 
discriminator_predictions.dense.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


KeyboardInterrupt: 

## Summary

In [ ]:
abl_df = pd.read_csv('../outputs/ablations/ablation_results.csv')

print('='*70)
print('ABLATION RESULTS  (BanglaBERT, seed=42)')
print('='*70)

show_cols = ['tag','binary_macro_f1','binary_accuracy','binary_mcc']
if 'abuse_type_macro_f1' in abl_df.columns:
    show_cols += ['abuse_type_macro_f1']
print(abl_df[show_cols].to_string(index=False))

# Deltas vs full_multitask
ref_bin = abl_df.loc[abl_df['tag']=='full_multitask','binary_macro_f1'].values
if len(ref_bin):
    ref_bin = ref_bin[0]
    abl_df['delta_binary'] = (abl_df['binary_macro_f1'] - ref_bin).round(4)
    print(f'\nΔ vs full_multitask (binary macro-F1, ref={ref_bin:.4f}):')
    print(abl_df[['tag','binary_macro_f1','delta_binary']].to_string(index=False))

if 'abuse_type_macro_f1' in abl_df.columns:
    ref_abu = abl_df.loc[abl_df['tag']=='full_multitask','abuse_type_macro_f1'].values
    if len(ref_abu):
        ref_abu = ref_abu[0]
        abl_df['delta_abuse'] = (abl_df['abuse_type_macro_f1'].fillna(0) - ref_abu).round(4)
        print(f'\nΔ vs full_multitask (abuse_type macro-F1, ref={ref_abu:.4f}):')
        print(abl_df[['tag','abuse_type_macro_f1','delta_abuse']].to_string(index=False))

abl_df.to_csv('../outputs/ablations/ablation_results.csv', index=False)
print('\n✅ ablation_results.csv saved → ../outputs/ablations/')


---
**Next:** Run `07_ablations_and_analysis.ipynb` last (CPU only, 2 min).